# 72 — Train SASRec recall channel + ablate vs union (P0)

Trains the dialog-conditioned, content-fused SASRec on Colab GPU and measures
recall@100 lift as a 4th channel in `wrrf_union_v1` (`use_sasrec`).

Run order:
1. Cell 1 — setup (clones the branch, installs deps, mounts Drive, symlinks the cache).
2. Cell 2 — train SASRec (writes `sasrec_v1/sasrec.pt` to the Drive cache).
3. Cell 3 — recall ablation (union vs union+SASRec on the full dev split).
4. Cell 4 — train content-fusion ablation variants (random id-proxy + metadata + audio).
5. Cell 5 — content-fusion ablation eval + SASRec fusion-weight sweep.

References: spec `docs/superpowers/specs/2026-05-27-sasrec-recall-channel-design.md`,
plan `docs/superpowers/plans/2026-05-27-sasrec-recall-channel.md`,
memory `project_sasrec_p0_implemented_2026_05_27.md`.


In [ ]:
# 1) Setup. Disable JAX GPU preallocation BEFORE any import pulls JAX in.
# datasets/transformers import JAX transitively; JAX grabs ~75% of VRAM on
# first use, so the KERNEL ends up hogging the GPU and the cell-3 !python
# subprocess OOMs. This is why nb 72 OOM'd while nb 70/71 (which set these)
# did not. If the kernel already imported JAX, RESTART RUNTIME for this to take.
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH', 'true')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

BRANCH = 'recall-union-lgbm'  # G2: 3-channel union pool + new session features + album_name fix
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
src = f'{DRIVE_BASE}/recsys2026_retrieval_v2_cache'
dst = f'{LOCAL_BASE}/retrieval_v2'
if os.path.islink(dst): os.unlink(dst)
elif os.path.exists(dst):
    import shutil; shutil.rmtree(dst)
os.symlink(src, dst)

# Retrieval stack (bm25->bm25s, dense->sentence-transformers/peft) is imported
# eagerly by mcrs.retrieval_modules, so its deps are required even for the
# LGBM build. Matches nb 71's proven set + lightgbm/scikit-learn.
!pip install -q --upgrade 'transformers>=4.40' 'accelerate>=0.30' 'peft>=0.11' \
    'datasets' 'pandas<3.0' 'tqdm' 'huggingface_hub' 'sentence-transformers>=3.0' \
    'FlagEmbedding>=1.3' 'bm25s' 'lightgbm' 'scikit-learn'

In [ ]:
# 2) Train SASRec on the train split (Colab GPU). Writes
# experiments/cache/retrieval_v2/sasrec/sasrec_v1/sasrec.pt -> Drive (via the
# cell-1 symlink), so it persists across runtimes.
#
# Per-epoch output: train_loss, val_loss, val_recall@{20,100} (val held out
# session-disjoint from train via SHA1(session_id)). End-of-run: standalone
# TEST recall@{20,100} (sanity number; model selection uses val, not test).
#
# Pre-flight — run ONCE to confirm the CLAP audio column name on the dataset.
# If it isn't 'audio-laion_clap', edit CLAP_COL in scripts/train_sasrec.py
# and re-commit + git pull on Colab.
#   from datasets import load_dataset
#   ds = load_dataset('talkpl-ai/TalkPlayData-Challenge-Track-Embeddings', split='all_tracks')
#   print([c for c in ds.column_names if 'clap' in c.lower() or 'audio' in c.lower()])
!cd /content/recsys2026 && python -u scripts/train_sasrec.py \
    --cache-dir /content/recsys2026/experiments/cache \
    --out sasrec_v1 \
    --epochs 10


In [ ]:
# 3) SASRec recall ablation: union without vs with the SASRec channel
# (use_sasrec). Builds dev data with the user-turns dialog, prints the
# dialog-length distribution vs bge-base-en's 512 cap, then reports
# recall@{20,100} for both unions plus the delta. Decision gate:
# union+SASRec recall@100 >= baseline + ~0.03 -> proceed to P1 (feed the
# SASRec score into the LGBM as the discriminating relevance feature).
import sys
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer
from mcrs.db_item.music_catalog import MusicCatalogDB
from mcrs.retrieval_modules import load_retrieval_module
from mcrs.retrieval_modules.sasrec_model import build_user_dialog

# Shared constants (self-contained — no dependency on earlier cells).
ITEM_DB = 'talkpl-ai/TalkPlayData-Challenge-Track-Metadata'
CORPUS = ['track_name', 'artist_name', 'album_name']
CACHE_DIR = '/content/recsys2026/experiments/cache'

item_db = MusicCatalogDB(ITEM_DB, ['all_tracks'], CORPUS)
dev = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')
queries, golds, user_ids, played, user_dialogs = [], [], [], [], []
N_SASREC_EVAL = None  # None = full dev; set an int to cap for a smoke run
for sess in dev:
    if N_SASREC_EVAL is not None and len(queries) >= N_SASREC_EVAL:
        break
    df = pd.DataFrame(sess['conversations'])
    for _, music in df[df['role'] == 'music'].iterrows():
        if N_SASREC_EVAL is not None and len(queries) >= N_SASREC_EVAL:
            break
        tn = int(music['turn_number'])
        prior = df[(df['turn_number'] < tn) |
                   ((df['turn_number'] == tn) & (df['role'] == 'user'))]
        lines = []
        for _, t in prior.iterrows():
            role = 'assistant' if t['role'] == 'music' else t['role']
            content = item_db.id_to_metadata(t['content']) if t['role'] == 'music' else t['content']
            lines.append(f'{role}: {content}')
        queries.append(chr(10).join(lines))
        user_dialogs.append(build_user_dialog(prior.to_dict('records')))
        golds.append(music['content'])
        user_ids.append(sess.get('user_id'))
        played.append(list(df[(df['role'] == 'music') & (df['turn_number'] < tn)]['content']))
print('[sasrec eval] built', len(queries), 'dev turns')

# Dialog-length check: how many user-dialogs exceed bge-base-en's 512-token cap?
tok = AutoTokenizer.from_pretrained('BAAI/bge-base-en-v1.5')
lens = [len(tok.encode(d)) for d in user_dialogs]
print('[sasrec eval] user-dialog tokens: median', int(np.median(lens)),
      ' p95', int(np.percentile(lens, 95)),
      ' frac>512:', round(float(np.mean([l > 512 for l in lens])), 4))

def recall_at(cands, k):
    return float(np.mean([1.0 if g in c[:k] else 0.0 for c, g in zip(cands, golds)]))

base = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS,
                             CACHE_DIR, extra_config={})
sas = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS,
                            CACHE_DIR, extra_config={'use_sasrec': True, 'w_sasrec': 1.0})
ctx = [{'history_tids': p, 'user_dialog': ud} for p, ud in zip(played, user_dialogs)]
cb = base.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
cs = sas.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
print('=== SASRec channel recall ablation (n=' + str(len(golds)) + ', FULL dev) ===')
print('  union (3-chan) : recall@20=' + str(round(recall_at(cb, 20), 4)) +
      ' @100=' + str(round(recall_at(cb, 100), 4)))
print('  union + SASRec : recall@20=' + str(round(recall_at(cs, 20), 4)) +
      ' @100=' + str(round(recall_at(cs, 100), 4)) + '   (G1 gate 0.46)')
print('  delta recall@100 :', round(recall_at(cs, 100) - recall_at(cb, 100), 4))


## 4-5 — content-fusion ablation + fusion-weight sweep

Cell 4 trains the item-representation variants; cell 5 evaluates them. Both
answer the two open questions from the G1 pass:

- Content vs id-only: is the val->dev gap closed by real content, or could a
  memorized per-item embedding do as well? `random` mode = seeded per-item
  vectors (an ID proxy). If `content` >> `random` on dev, content is buying
  cold-item recall and the lever is richer content.
- Fusion weight: SASRec standalone ~= the whole 3-chan union, so it may
  deserve more RRF mass than the default 1.0.

Cell 5 reuses RRF_MODEL.fuse_per_sub: the 3-chan base rankings are computed
ONCE and re-fused per variant/weight, so no retriever is re-run.

In [ ]:
# 4) Content-fusion ablation — train the item-representation variants.
# Axis = item feats source (apply_item_feats_mode). content (sasrec_v1) is
# already trained in cell 2. random is FIRST because content-vs-random is the
# core question; metadata/audio are the which-modality follow-up (comment them
# out for a faster pass). The ctx (dialog) cache is shared across modes, so
# only the training compute differs.
for mode, out in [('random', 'sasrec_random'),
                  ('metadata', 'sasrec_metadata'),
                  ('audio', 'sasrec_audio')]:
    print('\n========== training item-feats-mode=' + mode + ' -> ' + out + ' ==========')
    !cd /content/recsys2026 && python -u scripts/train_sasrec.py \
        --cache-dir /content/recsys2026/experiments/cache \
        --out {out} --item-feats-mode {mode} --epochs 10


In [ ]:
# 5) Content-fusion ablation eval + fusion-weight sweep. Requires cell 3
# (queries/ctx/golds/recall_at/base/user_ids) and cells 2+4 (trained variants).
# The 3-chan base rankings are computed once and re-fused with fuse_per_sub.
from mcrs.retrieval_modules.rrf import RRF_MODEL

base_per_sub, _ = base.batch_per_sub_rankings(
    queries, user_ids=user_ids, batch_context=ctx)
base_w = [s['weight'] for s in base.subs]
base_fused = RRF_MODEL.fuse_per_sub(base_per_sub, base_w, base.k, 100)
base100, base20 = recall_at(base_fused, 100), recall_at(base_fused, 20)

def _sasrec_rank(model_dir):
    r = load_retrieval_module('sasrec_seq', ITEM_DB, ['all_tracks'], CORPUS,
                              CACHE_DIR, extra_config={'model_dir': model_dir})
    return r.batch_text_to_item_retrieval(
        queries, topk=100, user_ids=user_ids, batch_context=ctx)

VARIANTS = [('content', 'sasrec_v1'), ('random(id-proxy)', 'sasrec_random'),
            ('metadata', 'sasrec_metadata'), ('audio', 'sasrec_audio')]
ranks = {}
print('=== content-fusion ablation: union+SASRec recall@100 (FULL dev) ===')
print('  3-chan (no SASRec) : @20=' + str(round(base20, 4)) +
      ' @100=' + str(round(base100, 4)))
for label, mdir in VARIANTS:
    try:
        sr = _sasrec_rank(mdir)
    except Exception as e:
        print('  + SASRec[' + label + '] : MISSING ' + mdir + ' - ' + repr(e)[:80])
        continue
    ranks[label] = sr
    fused = RRF_MODEL.fuse_per_sub(base_per_sub + [sr], base_w + [1.0], base.k, 100)
    r100 = recall_at(fused, 100)
    print('  + SASRec[' + label + '] : @100=' + str(round(r100, 4)) +
          '  delta=' + str(round(r100 - base100, 4)))

# Fusion-weight sweep on the CONTENT model (other channels held at their default).
print('\n=== SASRec fusion-weight sweep on content model (FULL dev) ===')
print('  w_sasrec=0.0 (base) : @20=' + str(round(base20, 4)) +
      ' @100=' + str(round(base100, 4)))
if 'content' in ranks:
    for w in [0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0]:
        fused = RRF_MODEL.fuse_per_sub(
            base_per_sub + [ranks['content']], base_w + [w], base.k, 100)
        print('  w_sasrec=' + str(w) + ' : @20=' + str(round(recall_at(fused, 20), 4)) +
              ' @100=' + str(round(recall_at(fused, 100), 4)))
